In [14]:
from ast import *
from utils import *
from x86_ast import *
import import_ipynb
from rco_test import *
from select_instr import *
from assign_homes import *


In [ ]:
def patch_instr(i: instr) -> List[instr]:
    match i:
        case Instr('movq',[Deref('rbp',num1),Deref('rbp',num2)]):
            return [Instr('movq',[Deref('rbp',num1),Reg('rax')]), 
                    Instr('movq',[Reg('rax'),Deref('rbp',num2)])]
        case Instr('addq', [Deref('rbp',num1), Deref('rbp',num2)]):
            return [Instr('movq',[Deref('rbp',num1),Reg('rax')]),
                    Instr('addq',[Reg('rax'),Deref('rbp',num2)])]
        case Instr('subq',[Deref('rbp',num1),Deref('rbp',num2)]):
            return [Instr('movq',[Deref('rbp',num1), Reg('rax')]),
                    Instr('subq',[Reg('rax'),Deref('rbp',num2)])]
        case Instr('addq',[Immediate(int), Deref('rbp',num)]):
            if Immediate(int).value > 2**16:
                return [Instr('movq',[Immediate(int), Reg('rax')]),
                        Instr('addq',[Reg('rax'),Deref('rbp',num)])]    
            return [i]
        case Instr('subq',[Immediate(int),Deref('rbp',num)]):       
            if Immediate(int).value > 2**16:
                return [Instr('movq',[Immediate(int),Reg('rax')]),
                        Instr('subq',[Reg('rax'),Deref('rbp',num)])]
            return [i]
        case Instr('movq',[Immediate(int), Deref('rbp',num)]):
            if Immediate(int).value > 2 ** 16:
                return [Instr('movq',[(Immediate(int),Reg('rax'))]),
                        Instr('movq',[Reg('rax'),Deref('rbp',num)])]
            return [i]
        case _:
            return [i]

In [16]:
def patch_instructions(p: X86Program) -> X86Program:
    new_list = []

    for instr in p.body:
                  new_list.extend(patch_instr(instr))
    new_program = X86Program(new_list)
    new_program.stack_space = p.stack_space
    return new_program

In [17]:
if __name__ == "__main__":
    import textwrap
    code = textwrap.dedent("""
    a = input_int() + input_int()
    b = a + 12
    c = 30
    c = c - b
    print(c)""")
    parsed_code = parse(code)
    rco_code = remove_complex_operands(parsed_code)
    select_instr_code = select_instruction(rco_code)
    assign_homes_code = assign_homes(select_instr_code)
    patch_instr_code = patch_instructions(assign_homes_code)
    print(select_instr_code)
    print(assign_homes_code)
    print(patch_instr_code)
    print(patch_instr_code.stack_space)

	.globl main
main:
      callq read_int
      movq %rax, temp.6
      callq read_int
      movq %rax, temp.7
      movq temp.6, %rax
      addq temp.7, %rax
      movq %rax, a
      movq a, %rax
      addq $12, %rax
      movq %rax, b
      movq $30, c
      subq b, c
      movq c, %rdi
      callq print_int


	.globl main
main:
      callq read_int
      movq %rax, -8(%rbp)
      callq read_int
      movq %rax, -16(%rbp)
      movq -8(%rbp), %rax
      addq -16(%rbp), %rax
      movq %rax, -24(%rbp)
      movq -24(%rbp), %rax
      addq $12, %rax
      movq %rax, -32(%rbp)
      movq $30, -40(%rbp)
      subq -32(%rbp), -40(%rbp)
      movq -40(%rbp), %rdi
      callq print_int


	.globl main
main:
      callq read_int
      movq %rax, -8(%rbp)
      callq read_int
      movq %rax, -16(%rbp)
      movq -8(%rbp), %rax
      addq -16(%rbp), %rax
      movq %rax, -24(%rbp)
      movq -24(%rbp), %rax
      addq $12, %rax
      movq %rax, -32(%rbp)
      movq $30, -40(%rbp)
      movq -32(